<a href="https://colab.research.google.com/github/Sizan99/ml-pipeline/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sizan99/ml-pipeline/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

I am using our honest Baseline Rule (Impressions × (1 - CTR)) instead of the ML model, because our ML-09 audit proved the Baseline was significantly more accurate and generalizable.

- **Archetype → Action Mapping:**
  - *Archetype:* "High Visibility, Low Engagement" (High Impressions, CTR < 1%)
  - *Action:* `REFRESH_SNIPPET` (Update the `<title>` and `<meta description>`)
- **Reason Code:** `massive_impressions_low_ctr`

In [ ]:
import pandas as pd
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
print("Loading March 2026 warehouse data...")
dataset_url = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
df_daily = pd.read_parquet(dataset_url, storage_options={"token": hf_token})

# Recreate our winning Baseline Rule
df = df_daily.groupby(['client_hash_id', 'content_hash_id']).agg(
    impressions=('gsc_impressions', 'sum'),
    clicks=('gsc_clicks', 'sum')
).reset_index()

df['ctr'] = (df['clicks'] / df['impressions']).fillna(0)
df['score'] = df['impressions'] * (1.0 - df['ctr'])

# Assign Actions and Reason Codes
df['action'] = 'REFRESH_SNIPPET'
df['reason_code'] = 'massive_impressions_low_ctr'

# Filter and rank
df_queue = df[(df['impressions'] >= 1000) & (df['ctr'] < 0.01)].sort_values(by='score', ascending=False)
print(f"Generated a prioritized queue of {len(df_queue):,} actionable pages.")


Loading March 2026 warehouse data...
Generated a prioritized queue of 43,266 actionable pages.


## 2. Intended use and limits

- **Intended Use:** This queue is a directional, decision-support tool for the SEO/Content team to prioritize their daily workflow. It tells them exactly which pages are bleeding the most potential traffic.
- **The Decay/Refresh Insight:** Search engine snippets naturally decay as competitors aggressively optimize their titles to steal clicks. Refreshing a title tag is the fastest way to combat this decay without having to rewrite the entire article.
- **Cost/Value Thinking:** Rewriting a title tag takes an editor roughly 5 minutes (Low Cost). If doing so lifts the CTR from 0.5% to 1.5% on a page getting 200,000 impressions, it yields 2,000 free clicks instantly (High Value).
- **Limits:** This score cannot read search intent. It does not know the difference between a bad snippet and a "zero-click" query (like "how many ounces in a cup") where the user gets the answer directly on the Google search page and doesn't need to click.

In [ ]:
# Documented in markdown above.

## 3. Human review + the no-go list

- **Human Review Required:** Before refreshing a snippet, an editor must manually search the page's top query on Google to verify if it is an informational "zero-click" search result. If it is, no action is needed.
- **The NO-GO List:** We must strictly prohibit this queue from being hooked into an LLM to automatically overwrite `<title>` tags or meta descriptions without human approval. LLMs can hallucinate or change the core tone of a brand's page, causing severe reputational damage.

In [ ]:
# Documented in markdown above.

## 4. Monitoring / retrain triggers

- **Monitoring:** We will track the 30-day post-refresh CTR of the pages our team updates from this queue.
- **Review Trigger:** If the average CTR of the refreshed pages does not lift by at least 10% compared to a control group, the queue logic has gone stale and the baseline rule must be reviewed.

In [ ]:
# Documented in markdown above.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
import os

# Create outputs directory
os.makedirs('work/outputs', exist_ok=True)
csv_path = 'work/outputs/action_playbook_queue.csv'

# Export the top 50 ranked actions for the research paper
df_queue.head(50).to_csv(csv_path, index=False)
print(f"Successfully exported the Top 50 ranked actions to {csv_path}!")


Successfully exported the Top 50 ranked actions to work/outputs/action_playbook_queue.csv!


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.